# Source Modification Example

As an example of a source code modification, we will add a new CAM diagnostic variable:

Adding a new diagnostic variable typically consists of four steps:

1. Compute the variable.
2. Register the field with the CAM history system using `addfld`.
3. Supply the field values using `outfld`.
4. (Optional) Add the field to the default history output using `add_default`.

**Example**: We will illustrate the step by adding atmospheric temperature at **750 hPa**, called **T750** to CAM. Although CAM outputs temperature at several standard pressure levels (for example, T500), it does not provide `T750` by default. We will add `T750` below. 


## Registering a field with `addfld`

Before a field can be written to a history file, it must be registered with the CAM history system using:

```fortran
call addfld(fname, type, avgflag, units, long_name)
```

where:

- `fname` — field name
- `type` — field dimensions (for example, `horiz_only` for a 2D field or `(/'lev'/)` for a 3D field)
- `avgflag` — averaging flag (`'A'` = averaged, `'I'` = instantaneous)
- `units` — field units
- `long_name` — descriptive name

For **T750**, the registration is:

```fortran
call addfld('T750', horiz_only, 'A', 'K', &
            'Temperature at 750 hPa pressure surface')
```

This registers **T750** as a two-dimensional, time-averaged field with units of Kelvin.

## Writing values with `outfld`

Once the field is registered, the computed values are passed to the history system using:

```fortran
call outfld(fname, field, idim, c)
```

where:

- `fname` — field name
- `field` — array containing the field values
- `idim` — horizontal dimension of the field
- `c` — chunk (physics) or latitude (dynamics) index

For example:

```fortran
call outfld('T750', t750, pcols, lchnk)
```

where `t750` contains the temperature interpolated to 750 hPa. The CAM history system then performs any requested time averaging and writes the field to the appropriate history file.

## Adding the field to the default history stream

Registering a field does **not** automatically include it in the default history output. To do so, call:

```fortran
call add_default(fname, tindex, flag)
```

where:

- `fname` — field name
- `tindex` — history stream index
- `flag` — averaging flag (`'A'` or `'I'`)

For example:

```fortran
call add_default('T750', 1, ' ')
```

This adds **T750** to the default CAM history stream (`h0`).

## Summary

Adding a new diagnostic variable to CAM consists of:

1. Computing the diagnostic quantity.
2. Registering it with `addfld`.
3. Writing the values with `outfld`.
4. Optionally adding it to the default history stream using `add_default`.

In the following exercise, we will add **T750**, rebuild CESM, and verify that the new variable appears in the CAM history files.